# 04 — Quantile / Heteroscedastic Regression + New Sensor

Predict **Pressure** from sibling sensors using:
- Quantile regression (pinball loss) → P10/P50/P90
- Heteroscedastic head (Gaussian NLL) → mean ± interval

**New sensor demo**: train without `Accelerometer2RMS`, then add it and compare interval width.

In [ ]:
import numpy as np

from sentinelai.config import NEW_SENSOR, REGRESSION_TARGET, SENSORS
from sentinelai.data.skab import load_all_scenarios
from sentinelai.data.windows import inject_gaps, make_windows, temporal_split
from sentinelai.models.heteroscedastic import predict_interval, train_heteroscedastic_regressor
from sentinelai.models.quantile import (
    build_regression_windows,
    predict_quantiles,
    train_quantile_regressor,
)
from sentinelai.policy.decision import regression_policy_from_interval

In [ ]:
target_idx = SENSORS.index(REGRESSION_TARGET)
new_sensor_idx = SENSORS.index(NEW_SENSOR)

# Input sensors: all except target; optionally exclude "new" sensor
base_inputs = [i for i, s in enumerate(SENSORS) if s != REGRESSION_TARGET and s != NEW_SENSOR]
extended_inputs = [i for i, s in enumerate(SENSORS) if s != REGRESSION_TARGET]
print("Base inputs:", [SENSORS[i] for i in base_inputs])
print("Extended (+ new sensor):", [SENSORS[i] for i in extended_inputs])

In [ ]:
df = load_all_scenarios(("valve1", "valve2"))
batch = make_windows(inject_gaps(df))
train, cal, test = temporal_split(batch)

x_train_base, y_train, _ = build_regression_windows(train, target_idx, base_inputs)
x_test_base, y_test, cov_test = build_regression_windows(test, target_idx, base_inputs)
x_train_ext, _, _ = build_regression_windows(train, target_idx, extended_inputs)
x_test_ext, _, cov_ext = build_regression_windows(test, target_idx, extended_inputs)

## Quantile regression

In [ ]:
q_base = train_quantile_regressor(x_train_base, y_train, epochs=15)
q_ext = train_quantile_regressor(x_train_ext, y_train, epochs=15)

pred_base = predict_quantiles(q_base.model, x_test_base, q_base.quantile_levels)
pred_ext = predict_quantiles(q_ext.model, x_test_ext, q_ext.quantile_levels)

width_base = (pred_base['upper'] - pred_base['lower']).mean()
width_ext = (pred_ext['upper'] - pred_ext['lower']).mean()
print(f"Mean interval width — base: {width_base:.3f}, +new sensor: {width_ext:.3f}")

## Heteroscedastic regression

In [ ]:
h_base = train_heteroscedastic_regressor(x_train_base, y_train, epochs=15)
h_ext = train_heteroscedastic_regressor(x_train_ext, y_train, epochs=15)

int_base = predict_interval(h_base.model, x_test_base)
int_ext = predict_interval(h_ext.model, x_test_ext)
print(f"Mean std — base: {int_base['std'].mean():.3f}, +new sensor: {int_ext['std'].mean():.3f}")

## Policy on regression intervals

Escalate when the **upper bound** exceeds a pressure limit; abstain when the interval is wide.

In [ ]:
PRESSURE_LIMIT = float(np.quantile(y_test, 0.95))
print(f"Demo limit (95th pct Pressure): {PRESSURE_LIMIT:.2f}")

actions_base = [
    regression_policy_from_interval(u, l, PRESSURE_LIMIT, c)
    for u, l, c in zip(pred_base['upper'], pred_base['lower'], cov_test)
]
actions_ext = [
    regression_policy_from_interval(u, l, PRESSURE_LIMIT, c)
    for u, l, c in zip(pred_ext['upper'], pred_ext['lower'], cov_ext)
]
print("Base:", {a: actions_base.count(a) for a in set(actions_base)})
print("+New sensor:", {a: actions_ext.count(a) for a in set(actions_ext)})

## Takeaway

Adding a correlated sensor narrows predictive intervals → fewer abstains and more confident (but still bounded) escalation decisions.